# GLM-5.2 GPTQ Calibration — Colab Smoke Test

Validates fix for [llm-compressor #2862](https://github.com/vllm-project/llm-compressor/issues/2862):
- `sequential_targets=['Linear']` → fx-trace failure on `GlmMoeDsaIndexer` (data-dependent top-k)
- Default whole-decoder-layer targets → OOM on 128-expert MoE
- 4D activations from attention pass → Hessian accumulation crash (fixed in `accumulate_hessian`)

**Fix:** `sequential_targets=['GlmMoeDsaAttention', 'ExpertMLP']` + 4D input flattening in GPTQ

| GPU | Model | Expected |
|-----|-------|----------|
| T4 (15 GB) | GLM-5.2-0.8B-A0.8B | ✅ fits |
| A100 (40/80 GB) | GLM-5.2 (9B) | ✅ fits |

Run cells top-to-bottom. The final cell runs an inference sanity-check on the quantized model.

## 0 — GPU check

In [ ]:
import subprocess, sys

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('No GPU detected. Runtime → Change runtime type → GPU')

gpu_info = result.stdout.strip()
print('GPU:', gpu_info)

# Pick model based on available VRAM
vram_mb = int(gpu_info.split(',')[1].strip().split()[0])
if vram_mb >= 35000:
    MODEL_ID = 'zai-org/GLM-5.2'
    print('A100 detected → using 9B model')
else:
    MODEL_ID = 'inference-optimization/GLM-5.2-0.8B-A0.8B'
    print('T4 detected → using 0.8B model')

print('Model:', MODEL_ID)

## 1 — Install dependencies

Installing from the PR branch to test the fix.

In [ ]:
!pip install -q \
    'git+https://github.com/jayakumarpujar/llm-compressor.git@fix/glm-moe-dsa-gptq-sequential-targets-clean' \
    'transformers>=5.9.0' \
    datasets \
    accelerate
print('Install complete')

## 2 — Imports & version check

In [ ]:
import torch
import transformers
import llmcompressor
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier
from llmcompressor.utils import load_context

print(f'torch         {torch.__version__}')
print(f'transformers  {transformers.__version__}')
print(f'llmcompressor {llmcompressor.__version__}')
print(f'CUDA          {torch.cuda.is_available()}')

## 3 — Unit test: trace failure vs success

Reproduces the two failure modes from issue #2862 on a tiny CPU model (no GPU memory needed).

In [ ]:
try:
    from transformers.models.glm_moe_dsa.configuration_glm_moe_dsa import GlmMoeDsaConfig
    from transformers.models.glm_moe_dsa.modeling_glm_moe_dsa import GlmMoeDsaForCausalLM
except ImportError:
    print('SKIP: transformers does not have glm_moe_dsa — update transformers and re-run')
    raise SystemExit(0)

from llmcompressor.args.dataset_arguments import DatasetArguments
from llmcompressor.modeling.moe.linearize import linearize_moe
from llmcompressor.pipelines.sequential.helpers import trace_subgraphs
from llmcompressor.utils.dev import skip_weights_initialize

TINY_CFG = dict(
    hidden_size=128, intermediate_size=256, moe_intermediate_size=64,
    num_hidden_layers=2, num_attention_heads=4, num_key_value_heads=4,
    num_local_experts=4, n_routed_experts=4, num_experts_per_tok=2,
    n_shared_experts=1, n_group=1, topk_group=1, vocab_size=256,
)

config = GlmMoeDsaConfig(**TINY_CFG)
with skip_weights_initialize():
    tiny_model = GlmMoeDsaForCausalLM(config)
linearize_moe(tiny_model)

sample_input = {'input_ids': torch.zeros(1, 8, dtype=torch.long)}
ignore = DatasetArguments().tracing_ignore

# ── Test 1: ["Linear"] must FAIL (GlmMoeDsaIndexer is untraceable) ──
try:
    trace_subgraphs(tiny_model, sample_input, sequential_targets=['Linear'], ignore=ignore)
    print('FAIL  ["Linear"] should have raised but did not')
except Exception as e:
    print(f'PASS  ["Linear"] correctly raises: {type(e).__name__}')

# ── Test 2: ["GlmMoeDsaAttention", "ExpertMLP"] must SUCCEED ──
try:
    subgraphs = trace_subgraphs(
        tiny_model, sample_input,
        sequential_targets=['GlmMoeDsaAttention', 'ExpertMLP'],
        ignore=ignore,
    )
    assert len(subgraphs) > 1, f'Expected >1 subgraphs, got {len(subgraphs)}'
    print(f'PASS  ["GlmMoeDsaAttention", "ExpertMLP"] traced → {len(subgraphs)} subgraphs')
except Exception as e:
    print(f'FAIL  trace raised unexpectedly: {e}')

## 4 — Unit test: 4D activation flattening in GPTQ

Verifies the `accumulate_hessian` fix handles 4D inputs without crashing.

In [ ]:
from llmcompressor.modifiers.gptq.gptq_quantize import accumulate_hessian, make_empty_hessian

linear = torch.nn.Linear(64, 32)
H = make_empty_hessian(linear)
num_samples = torch.tensor(0)

# 4D input: [batch, extra_dim, seq, hidden] — what GLM-5.2 attention can produce
inp_4d = torch.randn(2, 4, 8, 64)
try:
    H, num_samples = accumulate_hessian(inp_4d, linear, H, num_samples)
    print(f'PASS  4D input flattened correctly, H shape={tuple(H.shape)}')
except Exception as e:
    print(f'FAIL  4D input raised: {e}')

## 5 — Load model & tokenizer

In [ ]:
from llmcompressor.modeling.moe.linearize import linearize_moe

# load_context() fast-path fails for glm_moe_dsa: inherits glm4_moe conversion
# mappings → tries to set fused gate_up_proj on LinearExperts2D at load time.
# Use device_map='auto' to load directly onto GPU, then linearize in-place.
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype='auto', device_map='auto')
linearize_moe(model)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Loaded:', MODEL_ID)
total_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f'Parameters: {total_params:.2f}B')
print(f'Device: {next(model.parameters()).device}')

## 6 — Calibration dataset

In [ ]:
DATASET_ID = 'HuggingFaceH4/ultrachat_200k'
DATASET_SPLIT = 'train_sft'
NUM_CALIBRATION_SAMPLES = 512
MAX_SEQUENCE_LENGTH = 2048

ds = load_dataset(DATASET_ID, split=DATASET_SPLIT)
ds = ds.shuffle(seed=42).select(range(NUM_CALIBRATION_SAMPLES))

def preprocess(example):
    return {'text': tokenizer.apply_chat_template(example['messages'], tokenize=False)}

def tokenize(sample):
    return tokenizer(
        sample['text'],
        padding='max_length',
        max_length=MAX_SEQUENCE_LENGTH,
        truncation=True,
        add_special_tokens=False,
    )

ds = ds.map(preprocess)
ds = ds.map(tokenize, remove_columns=ds.column_names)
print(f'Dataset ready: {len(ds)} samples')

## 7 — GPTQ calibration

Core of the fix:
- `GlmMoeDsaAttention` → leaf boundary, hides untraceable `GlmMoeDsaIndexer` from fx tracer
- `ExpertMLP` → one Hessian per expert on GPU at a time, avoids OOM
- 4D activation flattening in `accumulate_hessian` handles GLM-5.2 attention outputs

In [ ]:
import gc
torch.cuda.empty_cache()
gc.collect()

ignore = [
    're:model.layers.0.*',
    're:model.layers.1.*',
    're:model.layers.2.*',
    'lm_head',
    're:.*mlp.gate$',
]

recipe = GPTQModifier(targets='Linear', scheme='W4A16', ignore=ignore)

num_experts = getattr(model.config, 'n_routed_experts', 384)
targets_per_subgraph = num_experts // 4 + 10
print(f'n_routed_experts={num_experts}, targets_per_subgraph={targets_per_subgraph}')

oneshot(
    model=model,
    dataset=ds,
    batch_size=4,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
    sequential_targets=['GlmMoeDsaAttention', 'ExpertMLP'],
    sequential_targets_per_subgraph=targets_per_subgraph,
)

print('Calibration complete')

## 8 — Peak VRAM used

In [ ]:
peak_mb = torch.cuda.max_memory_allocated() / 1024**2
total_mb = torch.cuda.get_device_properties(0).total_memory / 1024**2
print(f'Peak VRAM used: {peak_mb:.0f} MB / {total_mb:.0f} MB ({100*peak_mb/total_mb:.1f}%)')

## 9 — Save compressed model

In [ ]:
SAVE_DIR = MODEL_ID.rstrip('/').split('/')[-1] + '-W4A16-G128'
model.save_pretrained(SAVE_DIR, save_compressed=True)
tokenizer.save_pretrained(SAVE_DIR)
print('Saved to:', SAVE_DIR)

## 10 — Inference sanity check

In [ ]:
model.eval()
prompt = 'What is the capital of France?'
messages = [{'role': 'user', 'content': prompt}]
input_ids = tokenizer.apply_chat_template(
    messages, return_tensors='pt', add_generation_prompt=True
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(input_ids, max_new_tokens=64, do_sample=False)

response = tokenizer.decode(output_ids[0][input_ids.shape[1]:], skip_special_tokens=True)
print('Prompt :', prompt)
print('Response:', response)

## Summary

| Check | Cell |
|-------|------|
| `["Linear"]` trace correctly fails | 3 |
| `["GlmMoeDsaAttention", "ExpertMLP"]` trace succeeds | 3 |
| 4D activation input handled without crash | 4 |
| Full GPTQ calibration completes without OOM | 7 |
| Quantized model generates coherent output | 10 |

If all cells pass, the fix is validated. Report results back on [PR #2912](https://github.com/vllm-project/llm-compressor/pull/2912).